[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 归一化流（Normalizing flows）

下面的图片取自一篇关于归一化流的非常好的博客文章：[blogpost](https://lilianweng.github.io/lil-log/2018/10/13/flow-based-deep-generative-models.html)

![](https://raw.githubusercontent.com/dataflowr/website/master/modules/extras/flows/three-generative-models.png)

**基于流的生成模型**由一系列**可逆**变换构成。流模型的主要优点是：它显式地学习数据分布 $p(\mathbf{x})$，因此损失函数就是简单的负对数似然。

给定一个样本 $\mathbf{x}$ 和一个先验 $p(\mathbf{z})$，我们用待学习的可逆函数 $f$ 计算 $f(\mathbf{x}) = \mathbf{z}$。给定 $f$ 和先验 $p(\mathbf{z})$，利用变量替换公式可以计算证据 $p(\mathbf{x})$：
$$
\begin{align*}
\mathbf{z} &\sim p(\mathbf{z}), \mathbf{z} = f(\mathbf{x}), \\
p(\mathbf{x}) 
&= p(\mathbf{z}) \left\vert \det \dfrac{d \mathbf{z}}{d \mathbf{x}} \right\vert  
= p(f(\mathbf{x})) \left\vert \det \dfrac{\partial f(\mathbf{x})}{\partial \mathbf{x}} \right\vert
\end{align*}
$$
其中 $\dfrac{\partial f(\mathbf{x})}{\partial \mathbf{x}}$ 是 $f$ 的雅可比矩阵。
回顾一下：给定一个把 $n$ 维输入向量 $\mathbf{x}$ 映射到 $m$ 维输出向量的函数 $f: \mathbb{R}^n \mapsto \mathbb{R}^m$，这个函数的所有一阶偏导数组成的矩阵叫做**雅可比矩阵** $J_f$，其中第 $i$ 行第 $j$ 列的元素是 $(J_f(\mathbf{x}))_{ij} = \frac{\partial f_i(\mathbf{x})}{\partial x_j}$：
$$
{J_f(\mathbf{x})} = \begin{bmatrix}
\frac{\partial f_1(\mathbf{x})}{\partial x_1} & \dots & \frac{\partial f_1(\mathbf{x})}{\partial x_n} \\[6pt]
\vdots & \ddots & \vdots \\[6pt]
\frac{\partial f_m(\mathbf{x})}{\partial x_1} & \dots & \frac{\partial f_m(\mathbf{x})}{\partial x_n} \\[6pt]
\end{bmatrix}
$$
下面，我们将用神经网络参数化 $f$，并通过最大化 $\ln p(\mathbf{x})$ 来学习 $f$。更准确地说，给定数据集 $(\mathbf{x}_1,\dots,\mathbf{x}_n)$ 和一个由先验 $p(\mathbf{z})$ 与神经网络 $f$ 构成的模型，我们通过最小化下面的式子来优化 $f$ 的权重：
$$
-\sum_{i}\ln p(\mathbf{x_i}) = \sum_i -\ln p(f(\mathbf{x}_i)) -\ln\left\vert \det \dfrac{\partial f(\mathbf{x}_i)}{\partial \mathbf{x}} \right\vert.
$$

**我们需要确保 $f$ 始终可逆，而且行列式容易计算。**

## [用 Real NVP 做密度估计](https://arxiv.org/abs/1605.08803)
作者：Laurent Dinh、Jascha Sohl-Dickstein、Samy Bengio（2016）

[Real NVP](https://arxiv.org/abs/1605.08803) 使用的函数 $f$ 由堆叠的仿射耦合层构成。对于输入 $\mathbf{x}\in \mathbb{R}^D$，仿射耦合层产生输出 $\mathbf{y}\in\mathbb{R}^D$，定义如下（$d<D$）：
$$
\begin{align}
\label{eq:aff}
\mathbf{y}_{1:d} &= \mathbf{x}_{1:d}\\
\mathbf{y}_{d+1:D} &= \mathbf{x}_{d+1:D} \odot \exp\left(s(\mathbf{x}_{1:d})\right) +t(\mathbf{x}_{1:d}) ,
\end{align}
$$
其中 $s$（缩放）和 $t$（平移）是从 $\mathbb{R}^d$ 映射到 $\mathbb{R}^{D-d}$ 的神经网络，$\odot$ 是逐元素乘法。

对任意函数 $s$ 和 $t$，仿射耦合层都是可逆的：
\begin{align*}
\begin{cases}
\mathbf{y}_{1:d} &= \mathbf{x}_{1:d} \\ 
\mathbf{y}_{d+1:D} &= \mathbf{x}_{d+1:D} \odot \exp({s(\mathbf{x}_{1:d})}) + t(\mathbf{x}_{1:d})
\end{cases}
\Leftrightarrow 
\begin{cases}
\mathbf{x}_{1:d} &= \mathbf{y}_{1:d} \\ 
\mathbf{x}_{d+1:D} &= (\mathbf{y}_{d+1:D} - t(\mathbf{y}_{1:d})) \odot \exp(-s(\mathbf{y}_{1:d}))
\end{cases}
\end{align*}

仿射耦合层的雅可比矩阵是一个下三角矩阵：
\begin{align*}
J(\mathbf{x}) =  \frac{\partial \mathbf{y}}{\partial \mathbf{x}}=
\begin{bmatrix}
  \mathbb{I}_d & \mathbf{0}_{d\times(D-d)} \\[5pt]
  \frac{\partial \mathbf{y}_{d+1:D}}{\partial \mathbf{x}_{1:d}} & \text{diag}(\exp(s(\mathbf{x}_{1:d})))
\end{bmatrix}
\end{align*}
因此行列式就是对对角线上各项求积：
\begin{align*}
\left\vert\det(J(\mathbf{x}))\right\vert
= \prod_{j=1}^{D-d}\exp(s(\mathbf{x}_{1:d}))_j
= \exp\left(\sum_{j=1}^{D-d} s(\mathbf{x}_{1:d})_j\right)
\end{align*}
注意，我们不需要计算 $s$ 或 $t$ 的雅可比矩阵；要计算 $f^{-1}$，也不需要求 $s$ 或 $t$ 的逆（它们可能不存在！）。换句话说，我们可以为 $s$ 和 $t$ 取任意复杂的函数。

在一个仿射耦合层里，某些维度（通道）保持不变。为了确保所有输入都有机会被改变，模型在每一层反转顺序，让不同的分量保持不变。按照这种交替的模式，在一个变换层中保持不变的单元集合，总会在下一层被修改。

这可以用二进制掩码实现。首先，我们可以把缩放网络和平移网络扩展成从 $\mathbb{R}^D$ 到 $\mathbb{R}^D$ 的映射。然后取一个掩码 $\mathbf{b} = (1,\dots,1,0,\dots,0)$，其中有 $d$ 个 1，于是仿射层可以写成：
\begin{align*}
\mathbf{y} = \mathbf{x} \odot \exp\big((1-\mathbf{b}) \odot s(\mathbf{b} \odot \mathbf{x})\big) + (1-\mathbf{b}) \odot t(\mathbf{b} \odot \mathbf{x}).
\end{align*}
注意我们有
\begin{align*}
\ln \left\vert\det(J(\mathbf{x}))\right\vert = \sum_{j=1}^{D} \Big((1-\mathbf{b})\odot s(\mathbf{b} \odot \mathbf{x})\Big)_j,
\end{align*}
仿射层的逆为：
\begin{align*}
\mathbf{x} = \left( \mathbf{y} -(1-\mathbf{b}) \odot t(\mathbf{b} \odot \mathbf{y})\right)\odot \exp\left( -(1-\mathbf{b}) \odot s(\mathbf{b} \odot \mathbf{y})\right)
\end{align*}
现在我们在相邻的耦合层之间交替使用二进制掩码 $\mathbf{b}$。

注意，论文里给出的公式略有不同：
$$\mathbf{y} = \mathbf{b} \odot \mathbf{x} + (1 - \mathbf{b}) \odot \Big(\mathbf{x} \odot \exp\big(s(\mathbf{b} \odot \mathbf{x})\big) + t(\mathbf{b} \odot \mathbf{x})\Big),$$
但两个公式给出的结果是一样的！


# Real NVP 的实现


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from pylab import rcParams
rcParams['figure.figsize'] = 5, 4
rcParams['figure.dpi'] = 150

import torch
from torch import nn
from torch import distributions
from torch.nn.parameter import Parameter

from sklearn import datasets

In [ ]:
nets = lambda: nn.Sequential(nn.Linear(2, 256), nn.LeakyReLU(), nn.Linear(256, 256), nn.LeakyReLU(), nn.Linear(256, 2), nn.Tanh())
nett = lambda: nn.Sequential(nn.Linear(2, 256), nn.LeakyReLU(), nn.Linear(256, 256), nn.LeakyReLU(), nn.Linear(256, 2))
# 不带参数、返回一个 pytorch 模型的函数，dim(X) -> dim(X)

masks = torch.from_numpy(np.array([[0, 1], [1, 0], [0, 1], [1, 0], [0, 1], [1, 0]]).astype(np.float32))
# 大小为 number_of_coupling_layers x dim(X) 的 torch.Tensor

In [ ]:
masks.shape

In [ ]:
from torch import distributions
prior = distributions.MultivariateNormal(torch.zeros(2), torch.eye(2))

In [ ]:
# 你可以计算 logprob，也可以从你的分布中采样：
print(prior.log_prob(torch.Tensor([0,0])))
print(prior.sample((3,)))

In [ ]:
class RealNVP(nn.Module):
    def __init__(self, nets, nett, mask, prior):
        super(RealNVP, self).__init__()
        
        # 创建一个流
        # nets：一个返回 PyTorch 神经网络的函数，例如 nn.Sequential，s = nets()，s: dim(X) -> dim(X)
        # nett：一个返回 PyTorch 神经网络的函数，例如 nn.Sequential，t = nett()，t: dim(X) -> dim(X)
        # mask：大小为 #number_of_coupling_layers x #dim(X) 的 torch.Tensor
        # prior：一个 torch.distributions 里的对象，例如 torch.distributions.MultivariateNormal
        
        self.prior = prior
        self.mask = nn.Parameter(mask, requires_grad=False)
        self.t = torch.nn.ModuleList([nett() for _ in range(len(masks))])
        self.s = torch.nn.ModuleList([nets() for _ in range(len(masks))])
        
    def g(self, z):
        # 计算并返回 g(z) = x，
        #    其中 self.mask[i]、self.t[i]、self.s[i] 定义第 i 个带掩码的耦合层
        # z：形状为 batchSize x dim(X) 的 torch.Tensor
        # 返回 x：形状为 batchSize x dim(X) 的 torch.Tensor
        return x

    def f(self, x):        
        # 计算 f(x) = z 以及 f 的 log_det_Jakobian（对数雅可比行列式），
        #    其中 self.mask[i]、self.t[i]、self.s[i] 定义第 i 个带掩码的耦合层
        # x：形状为 batchSize x dim(X) 的 torch.Tensor，是一个数据点
        # 返回 z：形状为 batchSize x dim(X) 的 torch.Tensor，一个隐表示
        # 返回 log_det_J：长度为 batchSize 的 torch.Tensor
        
        return z, log_det_J
    
    def log_prob(self, x):
        # 计算并返回 log p(x)
        # 用变量替换公式和 f 计算出的 log_det_J
        # 返回 logp：长度为 batchSize 的 torch.Tensor
        return logp
        
    def sample(self, batchSize): 
        # 用 g 的实现从流中抽取并返回 batchSize 个样本
        # 返回 x：形状为 batchSize x dim(X) 的 torch.Tensor
        return x

In [ ]:
flow = RealNVP(nets, nett, masks, prior)

In [ ]:
# 验证流是可逆的 g(f(x)) = x  提示：torch.allclose

In [ ]:
optimizer = # optimizer = # 选择一个优化器，用 torch.optim 模块

for t in range(5001):    
    noisy_moons = datasets.make_moons(n_samples=100, noise=.05)[0].astype(np.float32)
    loss = # loss = # 计算最大似然损失
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if t % 500 == 0:
        print('iter %s:' % t, 'loss = %.3f' % loss)
        
# 验证损失在下降
# 下面的可视化看起来不错吗？

# 可视化


In [ ]:
rcParams['figure.figsize'] = 10, 8

画几幅图：
- 从流中采样的样本
- 从先验采样的样本
- 数据样本
- 从数据到先验的映射


In [ ]:
noisy_moons = datasets.make_moons(n_samples=1000, noise=.05)[0].astype(np.float32)
z = flow.f(torch.from_numpy(noisy_moons))[0].detach().numpy()
plt.subplot(221)
plt.scatter(z[:, 0], z[:, 1])
plt.title(r'$z = f(X)$')

z = np.random.multivariate_normal(np.zeros(2), np.eye(2), 1000)
plt.subplot(222)
plt.scatter(z[:, 0], z[:, 1])
plt.title(r'$z \sim p(z)$')

plt.subplot(223)
x = datasets.make_moons(n_samples=1000, noise=.05)[0].astype(np.float32)
plt.scatter(x[:, 0], x[:, 1], c='r')
plt.title(r'$X \sim p(X)$')

plt.subplot(224)
x = flow.sample(1000).detach().numpy()
plt.scatter(x[:, 0], x[:, 1], c='r')
plt.title(r'$X = g(z)$')

画出估计的密度：


In [ ]:
xpoints = np.linspace(-1.5, 2.5, 500)
ypoints = np.linspace(-1.0, 1.5, 500)
(x1, x2,) = np.meshgrid(xpoints, ypoints)
xgrid = np.concatenate((x1.reshape(-1, 1), x2.reshape(-1, 1)), axis=1).astype(np.float32)
p = np.exp(flow.log_prob(torch.from_numpy(xgrid)).detach().numpy())

In [ ]:
fig = plt.figure()
plt.imshow(
    p.reshape(x1.shape), aspect="equal", origin="lower")
plt.axis('off')
plt.show()

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)